# MIRO AI -- Benchmark de precision

objectif : heberger un LLM local sur le GPU gratuit de Kaggle, l'exposer via un tunnel public, puis pointer le pipeline MIRO AI

## 1. Verifier le GPU

In [36]:
!nvidia-smi

Mon Oct  5 11:56:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Installer Ollama

In [37]:
# zstd requis par l'installeur pour l'extraction ; pciutils (lspci) requis pour la détection GPU 
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%##########                       70.8%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [38]:
# 2bis. configuration OLLAMA_MODELS
import os, glob

OLLAMA_MODELS_DIR = "/kaggle/working/ollama_models"
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
READONLY_MODELS = False
print(f"{OLLAMA_MODELS_DIR} (téléchargement nécessaire ci-dessous).")

os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR

/kaggle/working/ollama_models (téléchargement nécessaire ci-dessous).


## 3. Demarrer le serveur Ollama en arrière-plan

In [ ]:
import subprocess, time, requests

# Tuer tout ollama serve deja lancé sinon le nouveau process echoue au bind du port 11434 silencieusement.
!pkill -f "ollama serve" || true
time.sleep(2)  # laisser le temps au port de se liberer avant le nouveau bind

os.environ["OLLAMA_ORIGINS"] = "*" # désactive le controle CORS sur le header Origin. 

# la fenêtre de contexte par defaut d'Ollama (4096 tokens,prompt+completion CUMULES) coupe la génération en cours de synthese sur les questions multi-hop
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"
os.environ["OLLAMA_LOAD_TIMEOUT"] = "30m"

OLLAMA_LOG_PATH = "/kaggle/working/ollama.log"  # un PIPE jamais lu se remplit et bloque le serveur ; le fichier garde aussi les erreurs des runners
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(OLLAMA_LOG_PATH, "w"), stderr=subprocess.STDOUT,
    env=os.environ,  # necessaire pour que OLLAMA_MODELS et OLLAMA_KEEP_ALIVE soient pris en compte
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama prêt.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama n'a pas demarré à temps -- lire OLLAMA_LOG_PATH.")

In [40]:
# 3bis. Verifier qu'Ollama a bien detecté le GPU
!ollama --version
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

]11;?\ollama version is 0.35.1
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB
Tesla T4, 0 MiB, 15360 MiB


## 4. Télécharger les modèles

In [41]:
MODEL = "qwen3:14b"

if not READONLY_MODELS:
    !ollama pull {MODEL}
else:
    print(f"{MODEL} supposé déja présent dans {OLLAMA_MODELS_DIR} -- vérification via 'ollama list' :")
    !ollama list

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling a8cc1361f314: 100% ▕██████████████████▏ 9.3 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 78b3b822087d: 100% ▕██████████████████▏  488 B                         
verifying sha256 digest 
writing manifest 
success 


In [42]:
# 4ter. Télécharger le modele d'embedding
EMBED_MODEL = "bge-m3"

if not READONLY_MODELS:
    !ollama pull {EMBED_MODEL}
else:
    print(f"{EMBED_MODEL} supposé deja présent dans {OLLAMA_MODELS_DIR}.")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling daec91ffb5dd:   0% ▕                  ▏ 5.3 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:   3% ▕                  ▏  35 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:  11% ▕██                ▏ 130 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:  21% ▕███               ▏ 238 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:  25% ▕████              ▏ 285 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:  35% ▕██████            ▏ 400 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:  44% ▕███████           ▏ 508 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:  48% ▕████████          ▏ 552 MB/1.2 GB                  pulling manifest 
pulling daec91ffb5dd:  58% ▕██████████        ▏ 669 MB/1.2 GB                  pulling manifest 
pullin

## 5. Préchauffage + sanity check tool-calling

In [ ]:
# 5. Prechauffer le chat et l'embedding en VRAM avant le sanity check
# Ollama instancie un runner different selon la presence de 'tools'
import requests, time

OLLAMA_URL = "http://localhost:11434"

_t0 = time.time()
requests.post(
    f"{OLLAMA_URL}/v1/chat/completions",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "hi"}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Renvoie la météo pour une ville donnée.",
                "parameters": {
                    "type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                },
            },
        }],
        "stream": False,
    },
    timeout=600,
).raise_for_status()
print(f"Chat charge en VRAM ({time.time() - _t0:.1f}s).")

_t0 = time.time()
requests.post(
    f"{OLLAMA_URL}/v1/embeddings",
    json={"model": EMBED_MODEL, "input": "hi"},
    timeout=600,
).raise_for_status()
print(f"Embedding charge en VRAM ({time.time() - _t0:.1f}s).")

loaded = [entry["name"] for entry in requests.get(f"{OLLAMA_URL}/api/ps", timeout=30).json()["models"]]
print("Modeles en VRAM :", loaded)
assert any(name.startswith(MODEL) for name in loaded), f"{MODEL} a ete decharge."
assert any(name.startswith(EMBED_MODEL) for name in loaded), (
    f"{EMBED_MODEL} a decharge le chat : verifier OLLAMA_MAX_LOADED_MODELS (cellule 7)."
)

In [44]:
# 5bis. Sanity check local -- verifier que le tool-calling natif fonctionne

import requests, json as _json

test_payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Quel temps fait-il a Paris ?"}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Renvoie la meteo pour une ville donnée.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }],
    "stream": False,
}

resp = requests.post("http://localhost:11434/v1/chat/completions", json=test_payload, timeout=300)  # large par securité, le modèle est déja préchauffé
resp.raise_for_status()
data = resp.json()
tool_calls = data["choices"][0]["message"].get("tool_calls")
print("Tool calls recus :", _json.dumps(tool_calls, indent=2, ensure_ascii=False))
assert tool_calls, "tool-calling natif KO, ne pas continuer avec ce modèle."
print("tool-calling natif confirmé.")

Tool calls recus : [
  {
    "id": "call_577j1o3i",
    "index": 0,
    "type": "function",
    "function": {
      "name": "get_weather",
      "arguments": "{\"city\":\"Paris\"}"
    }
  }
]
tool-calling natif confirmé.


## 6. Exposer le serveur via un tunnel public**Sécurité :** le tunnel rend l'API Ollama accessible depuis Internet et Ollama n'authentifie pas les requêtes avec `LOCAL_LLM_API_KEY=ollama`. Le tunnel est désactivé par défaut. Ne l'activez que pour un test temporaire avec des données non sensibles ; ne l'utilisez pas pour exposer une instance Odoo réelle ou des données métier. `OLLAMA_ORIGINS`/CORS n'est pas un mécanisme d'authentification.

In [45]:
ENABLE_PUBLIC_TUNNEL = FalseUSE_NGROK = Trueif not ENABLE_PUBLIC_TUNNEL:    print("Tunnel public désactivé. Activez ENABLE_PUBLIC_TUNNEL uniquement pour un test non sensible.")elif USE_NGROK:    !pip install -q pyngrok    from pyngrok import ngrok, conf    from kaggle_secrets import UserSecretsClient    user_secrets = UserSecretsClient()    NGROK_AUTHTOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")    conf.get_default().auth_token = NGROK_AUTHTOKEN    # host_header="localhost:11434" (rewrite) : Ollama rejette en 403 silencieux toute requete    # dont le header Host ne correspond pas a localhost/127.0.0.1 (protection anti DNS-rebinding,    # verification distincte du controle CORS Origin couvert par OLLAMA_ORIGINS).    public_url = ngrok.connect(11434, "http", host_header="localhost:11434")    print("URL publique ngrok (temporaire, valable tant que ce notebook tourne) :")    print(public_url)else:    print("USE_NGROK=False -- cloudflared reste le tunnel actif.")    # Tunnel via cloudflared (aucun compte requis)    import subprocess, re, time        !curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64    !chmod +x /usr/local/bin/cloudflared    cloudflared_proc = subprocess.Popen(        ["cloudflared", "tunnel", "--url", "http://localhost:11434",         "--http-host-header", "localhost:11434"],        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,    )    public_url = None    _t0 = time.time()    for line in cloudflared_proc.stdout:        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)        if m:            public_url = m.group(0)            break        if time.time() - _t0 > 60:            break    if public_url:        print("URL publique :")        print(public_url)    else:        raise RuntimeError("URL cloudflared non trouvée dans les logs apres 60s -- verifier cloudflared_proc.stdout manuellement.")

URL publique ngrok (temporaire, valable tant que ce notebook tourne) :
NgrokTunnel: "https://pellet-majority-lesser.ngrok-free.dev" -> "http://localhost:11434"


## 7. où tourne le pipeline MIRO AIUniquement après activation volontaire du tunnel dans la cellule 6, copiez son URL temporaire dans le `.env`. L'URL change à chaque session.**Ne connectez pas MIRO à une instance Odoo réelle ni à des données sensibles via ce tunnel public : la clé `ollama` n'est pas une authentification.** Pour un test, utilisez un environnement et des données non sensibles.Dans le `.env` :```LLM_PROVIDER=ollamaLOCAL_LLM_BASE_URL=<URL_TUNNEL_CI_DESSUS>/v1LOCAL_LLM_MODEL=<MODEL, cellule 4>LOCAL_LLM_API_KEY=ollamaEMBEDDING_BACKEND=ollamaLOCAL_EMBEDDING_BASE_URL=<URL_TUNNEL_CI_DESSUS>/v1LOCAL_EMBEDDING_MODEL=<EMBED_MODEL, cellule 4ter>LOCAL_EMBEDDING_API_KEY=ollamaLOCAL_ALLOWED_HOSTS=localhost,127.0.0.1,::1,<HOTE_DU_TUNNEL>```L'URL change a chaque session du notebook : mettre a jour les trois valeurs a chaque redemarrage.